# RQ2 — Dataset Construction: Semantics-Preserving Perturbations
This notebook loads the **test split** of the SPARQL-Embeddings dataset and applies
5 semantics-preserving perturbations to every test query, producing `(original, perturbed)` pairs.

**Perturbations applied:**
1. Variable Renaming
2. Triple Reordering
3. OPTIONAL Insertion
4. LIMIT/OFFSET Injection
5. Combined (all four above)

> ⚠️ Each generated pair must later be **validated** by executing both queries against the endpoint
> and confirming Answer-Set Jaccard = 1.0. A `validated` flag column is included for this step.

In [ ]:
import json
import re
import random
import copy
import hashlib
import pandas as pd
from pathlib import Path
from urllib.request import urlopen

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_RAW = "../dataset"

DATASETS = {
    "wikidata": {
        "queries_url" : f"{BASE_RAW}/wikidata/wikidata_queries.json",
        "splits_url"  : f"{BASE_RAW}/splits/splits_wikidata.json",
    },
    "dbpedia": {
        "queries_url" : f"{BASE_RAW}/dbpedia/dbpedia_queries.json",
        "splits_url"  : f"{BASE_RAW}/splits/splits_dbpedia.json",
    },
}

OUTPUT_DIR = Path("rq2_dataset")
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)

print("✅ Configuration loaded.")

In [ ]:
def load_json_url(url: str):
    """Fetch JSON from a raw GitHub URL."""
    with urlopen(url) as r:
        return json.loads(r.read().decode())

def get_test_queries(queries: list, splits: dict) -> list:
    """
    Return the subset of queries whose index is in the test split.
    Supports both list-of-indices and list-of-query-id splits.
    """
    test_ids = set(splits.get("test", splits.get("test_idx", [])))
    # If splits store integer indices
    if test_ids and isinstance(next(iter(test_ids)), int):
        return [q for i, q in enumerate(queries) if i in test_ids]
    # If splits store query ids / strings
    id_key = "id" if queries and "id" in queries[0] else list(queries[0].keys())[0]
    return [q for q in queries if q[id_key] in test_ids]

print("✅ Loader helpers defined.")

In [ ]:
test_queries_all = {}   # dataset_name -> list of query dicts

for ds_name, cfg in DATASETS.items():
    print(f"Loading {ds_name}...")
    queries = load_json_url(cfg["queries_url"])
    splits  = load_json_url(cfg["splits_url"])
    test_q  = get_test_queries(queries, splits)
    test_queries_all[ds_name] = test_q
    print(f"  → {len(test_q)} test queries")

print("\n✅ All data loaded.")

In [ ]:
# ── 1. Variable Renaming ──────────────────────────────────────────────────────
def rename_variables(query: str) -> str:
    """
    Replace every SPARQL variable (?name) with positional placeholders
    ?v0, ?v1, ... in order of first appearance.
    """
    seen = {}
    counter = [0]
    def replacer(m):
        orig = m.group(0)          # e.g. ?city
        if orig not in seen:
            seen[orig] = f"?v{counter[0]}"
            counter[0] += 1
        return seen[orig]
    return re.sub(r'\?[A-Za-z_][A-Za-z0-9_]*', replacer, query)


# ── 2. Triple Reordering ──────────────────────────────────────────────────────
def reorder_triples(query: str) -> str:
    """
    Shuffle the triple patterns inside the WHERE { } clause.
    Handles only flat BGPs (no sub-selects / OPTIONAL blocks inside).
    """
    where_match = re.search(r'(WHERE\s*\{)([^{}]*)(\})', query, re.IGNORECASE | re.DOTALL)
    if not where_match:
        return query
    prefix   = where_match.group(1)
    body     = where_match.group(2)
    suffix   = where_match.group(3)
    # Split on '.' keeping other lines (FILTER, BIND) in place
    triples  = [t.strip() for t in body.split('.') if t.strip()]
    if len(triples) < 2:
        return query              # nothing to reorder
    rng = random.Random(SEED)
    rng.shuffle(triples)
    new_body = ' . '.join(triples)
    return query[:where_match.start()] + prefix + ' ' + new_body + ' ' + suffix + query[where_match.end():]


# ── 3. OPTIONAL Insertion ─────────────────────────────────────────────────────
def insert_optional(query: str) -> str:
    """
    Append a dummy OPTIONAL { ?s <urn:sparql:dummy> ?o } block that always
    evaluates to empty, leaving the result set unchanged.
    """
    dummy = ' OPTIONAL { ?_dummy_s <urn:sparql:embeddings:dummy> ?_dummy_o } '
    # Insert just before the last closing brace of WHERE
    last_brace = query.rfind('}')
    if last_brace == -1:
        return query
    return query[:last_brace] + dummy + query[last_brace:]


# ── 4. LIMIT / OFFSET Injection ───────────────────────────────────────────────
def inject_limit_offset(query: str) -> str:
    """
    Append LIMIT 100000 OFFSET 0 when no LIMIT clause is already present.
    A very high LIMIT preserves the full result set.
    """
    if re.search(r'\bLIMIT\b', query, re.IGNORECASE):
        return query              # already has LIMIT — skip
    return query.rstrip() + ' LIMIT 100000 OFFSET 0'


# ── 5. Combined ───────────────────────────────────────────────────────────────
def combined_perturbation(query: str) -> str:
    q = rename_variables(query)
    q = reorder_triples(q)
    q = insert_optional(q)
    q = inject_limit_offset(q)
    return q


PERTURBATIONS = {
    "var_rename"   : rename_variables,
    "triple_reorder": reorder_triples,
    "optional_insert": insert_optional,
    "limit_offset"  : inject_limit_offset,
    "combined"      : combined_perturbation,
}

print("✅ All 5 perturbation functions defined.")

In [ ]:
def get_query_text(q: dict) -> str:
    """Extract the SPARQL string from a query dict (handles different key names)."""
    for key in ("query", "sparql", "sparqlQuery", "text"):
        if key in q:
            return q[key]
    return next(iter(q.values()))  # fallback: first value

def get_query_id(q: dict, idx: int) -> str:
    for key in ("id", "qid", "query_id"):
        if key in q:
            return str(q[key])
    return f"q_{idx}"


all_pairs = []   # flat list of dicts for the final CSV/JSON

for ds_name, test_qs in test_queries_all.items():
    for idx, q_dict in enumerate(test_qs):
        qid    = get_query_id(q_dict, idx)
        orig   = get_query_text(q_dict)

        for pert_name, pert_fn in PERTURBATIONS.items():
            perturbed = pert_fn(orig)

            # Skip if perturbation produced no change (e.g. LIMIT already present)
            if perturbed.strip() == orig.strip():
                status = "skipped_no_change"
            else:
                status = "pending_validation"   # needs endpoint check

            all_pairs.append({
                "dataset"           : ds_name,
                "query_id"          : qid,
                "perturbation"      : pert_name,
                "original_query"    : orig,
                "perturbed_query"   : perturbed,
                "ground_truth_jaccard": 1.0,   # by construction
                "validation_status" : status,   # fill after endpoint execution
                "pair_id"           : hashlib.md5(f"{ds_name}_{qid}_{pert_name}".encode()).hexdigest()[:12],
            })

df_pairs = pd.DataFrame(all_pairs)
print(df_pairs[["dataset","query_id","perturbation","validation_status"]].groupby(
    ["dataset","perturbation","validation_status"]).size().to_string())
print(f"\n✅ Total pairs generated: {len(df_pairs)}")

In [ ]:
# Quick sanity-check: show one original vs perturbed for each perturbation type
sample = df_pairs[df_pairs["dataset"] == "wikidata"].groupby("perturbation").first().reset_index()

for _, row in sample.iterrows():
    print(f"\n{'='*60}")
    print(f"Perturbation : {row['perturbation']}")
    print(f"Query ID     : {row['query_id']}")
    print(f"ORIGINAL:\n{row['original_query'][:300]}")
    print(f"\nPERTURBED:\n{row['perturbed_query'][:300]}")
    print(f"Status       : {row['validation_status']}")

In [ ]:
# ── Save as CSV (easy to share / inspect) ────────────────────────────────────
csv_path = OUTPUT_DIR / "rq2_pairs.csv"
df_pairs.to_csv(csv_path, index=False)
print(f"Saved CSV  → {csv_path}  ({len(df_pairs)} rows)")

# ── Save as JSON (preserves multi-line queries cleanly) ───────────────────────
json_path = OUTPUT_DIR / "rq2_pairs.json"
with open(json_path, "w") as f:
    json.dump(all_pairs, f, indent=2, ensure_ascii=False)
print(f"Saved JSON → {json_path}")

# ── Per-perturbation split files (handy for batch endpoint execution) ─────────
for pert_name, grp in df_pairs.groupby("perturbation"):
    p = OUTPUT_DIR / f"rq2_{pert_name}.json"
    grp.to_json(p, orient="records", indent=2, force_ascii=False)
    print(f"  → {p}  ({len(grp)} pairs)")

print("\n✅ All outputs saved.")

## (Optional) Endpoint Validation

After generating pairs you should **verify** each pair by executing both queries
against the live endpoint and checking that the Answer-Set Jaccard = 1.0.
The cell below shows the pattern — fill in your endpoint URLs and credentials.

```python
from SPARQLWrapper import SPARQLWrapper, JSON

ENDPOINTS = {
    "wikidata" : "https://query.wikidata.org/sparql",
    "dbpedia"  : "https://dbpedia.org/sparql",
}

def get_result_set(endpoint_url, query):
    sw = SPARQLWrapper(endpoint_url)
    sw.setQuery(query)
    sw.setReturnFormat(JSON)
    results = sw.query().convert()
    return frozenset(
        tuple(sorted(b.values())) for b in results["results"]["bindings"]
    )

def jaccard(set_a, set_b):
    if not set_a and not set_b: return 1.0
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union else 0.0

validated = []
for row in df_pairs.to_dict(orient="records"):
    ep  = ENDPOINTS[row["dataset"]]
    r_orig = get_result_set(ep, row["original_query"])
    r_pert = get_result_set(ep, row["perturbed_query"])
    j = jaccard(r_orig, r_pert)
    row["ground_truth_jaccard"] = j
    row["validation_status"] = "valid" if j == 1.0 else "DISCARD"
    validated.append(row)

df_validated = pd.DataFrame(validated)
df_validated = df_validated[df_validated["validation_status"] == "valid"]
df_validated.to_csv(OUTPUT_DIR / "rq2_pairs_validated.csv", index=False)
print(f"Valid pairs kept: {len(df_validated)}")
```

In [ ]:
print("RQ2 Dataset Summary")
print("=" * 50)
summary = df_pairs.groupby(["dataset","perturbation"]).agg(
    total_pairs   = ("pair_id", "count"),
    skipped       = ("validation_status", lambda x: (x=="skipped_no_change").sum()),
    pending       = ("validation_status", lambda x: (x=="pending_validation").sum()),
).reset_index()
print(summary.to_string(index=False))